# 06 — End-to-end inference

Putting all the pieces together: upload a real image, run the full pipeline, see boxes, see costs.

This notebook requires the **production weights** to be present locally. If you ran `app.py` once (or pointed the cells below at the GitHub Release tarball), they will be at `checkpoints/production/`. Otherwise the cell that creates the pipeline will say "checkpoint missing" and explain how to fetch them.


In [ ]:
# === Colab bootstrap ===
# Safe to re-run. On a local clone with `pip install -e .` already done this
# is a no-op; on Colab it clones the repo + installs deps the first time.
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/theDocWho/car-crash-fix-amount-predictor.git"
REPO_DIR = Path("car-crash-fix-amount-predictor")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
if IN_COLAB:
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)

# Make `ccdp` importable whether or not the package was installed editable.
src_path = Path("src").resolve()
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("ccdp path:", src_path)
print("running in Colab:", IN_COLAB)


## Fetch the production weights (once)

Same logic that runs on first boot of the HuggingFace Space.


In [ ]:
# Reuse the bootstrap from app.py — safe to run inside the notebook directory.
import sys
sys.path.insert(0, ".")
import app   # this triggers the weight + catalog bootstrap as a side-effect
print("Weights at:", "checkpoints/production/")


## Run the full pipeline on a sample image


In [ ]:
from pathlib import Path
from PIL import Image
from ccdp.preprocess import preprocess
from ccdp.viz import annotate_prediction
from ccdp.infer.variant_a import VariantAPipeline
from ccdp.infer.variant_b import VariantBPipeline

pipe_a = VariantAPipeline()
pipe_b = VariantBPipeline()

# Pick the first image you can find. Replace with your own path:
sample = next(Path("data").rglob("*.jpg"), None) or next(Path(".").rglob("*.png"), None)
print("sample:", sample)

img = Image.open(sample).convert("RGB")
img_clean, prep = preprocess(img)

pred_a = pipe_a.predict(img_clean).to_dict()
pred_b = pipe_b.predict(img_clean)

print("Variant A cost:", pred_a["cost"], pred_a["currency"])
print("Variant B cost:", pred_b.cost, pred_b.currency)
print("Detections   :", [(d.damage_type, round(d.confidence,2)) for d in pred_b.detections])

annotated = annotate_prediction(img_clean, pred_b)
annotated


That's everything: data in, boxes out, cost out. Modify the snippet above with your own image to verify on something not in the training set.

If you want to run the Gradio UI locally, from the repo root:

```bash
python app.py
```

Or the FastAPI server:

```bash
uvicorn ccdp.api.server:app --reload
```
